- Version 152
    - Keep `euros_per_mwh_lag_7_day_mean_diff` feature


- Version 151
    - eic_count/installed capacity lag features: keep NA instead of fill(0)
    
- Version 150
    - electricity price features: fix a typo in getting lag elec price (24 hours per day not 28)

-----
...


-----

- v69 `LB=74.32`
    - Model
        - Include almost all features
        - paramter change :  `cat_l2=0.1`, `num_leaves=256`, `stopping_rounds=50`, `reg_sqrt=rue`

- v65 `LB=74.94`
    - Model
        - Data preprocessing pipeline: Add routine to fill missing time.
        - Added new features `lag_2_day_consumption_counterpart`, `lag_7_day_mean_consumption_counterpart`, `euros_per_mwh`,
        - paramter change :  `stopping_rounds=100`

- v60 `LB=74.40`
    - Model
        - Changed base_lag to 48 hours


- v59 `LB=75.59`
    - Model
        - Removed forecast filter for `hour_ahead`
        - paramter change :  `cat_l2=0.1`


- v56 `LB=75.73`
    - Model
        - seperate models & features for is_consumption
        - cv_agg's l1: 29.0241 + 12.5268, cv_agg's l1: 47.618 + 4.03412
        - 



- v51 `LB=76.53`
    - Model
        - cv_agg's l1: 39.5448 + 7.62686
        - Added feature 'daytype`
        - parameter chagne: `cat_l2=1`, `reg_sqrt=True`
    - Analysis
        - Inspect errors by feature values


- v35 `LB=78.1`
    - Tweedie loss
    - Removed features: `sin_month`, `cos_month`
    - Added features: `lag_1_day_1h`, `lag_1_day_1h`, `lag_1_day_2h`, `lag_1_day_3h`


- v34 `LB 81.8`
    - Fix lag features for test time inference


In [1]:
# %load_ext cudf.pandas
import pandas as pd
# import cudf as pd

In [2]:
from sklearn.model_selection import TimeSeriesSplit, GroupKFold
from sklearn.metrics import mean_absolute_error as MAE
# from sklearn.metrics import mean_absolute_error

# from xgboost import XGBRegressor
# import catboost as cb
import lightgbm as lgb

import datetime

In [3]:
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

In [4]:
comp_path = "/kaggle/input/predict-energy-behavior-of-prosumers"
# comp_path = "./predict-energy-behavior-of-prosumers"

import json
with open(comp_path + "/county_id_to_name_map.json") as file:
    county_codes = json.load(file)

# Data Processing

In [5]:
import numpy as np
train = pd.read_csv(comp_path + "/train.csv")
elec_prices = pd.read_csv(comp_path + "/electricity_prices.csv")
gas_prices_train = pd.read_csv(comp_path +  "/gas_prices.csv")
client_train = pd.read_csv(comp_path + "/client.csv")


county_coordinate_weight_path = "/kaggle/input/enefit-county-coordinate-weights"
# county_coordinate_weight_path = "./enefit-county-coordinate-weights"

county_coordinate_weights = pd.read_csv(county_coordinate_weight_path + "/county_pixel_weights.csv")

train.datetime = pd.to_datetime(train.datetime)
train['date'] = pd.to_datetime(train.datetime).dt.date
# train['date'] = pd.to_datetime(train.datetime).dt.floor("D") # workaround for cudf


# elec_prices.forecast_date = pd.to_datetime(elec_prices.forecast_date).dt.date
# gas_prices.forecast_date = pd.to_datetime(gas_prices.forecast_date).dt.date
train.head(2)

   county  is_business  product_type  target  is_consumption   datetime  \
0       0            0             1   0.713               0 2021-09-01   
1       0            0             1  96.590               1 2021-09-01   

   data_block_id  row_id  prediction_unit_id        date  
0              0       0                   0  2021-09-01  
1              0       1                   0  2021-09-01  

In [6]:
client_train['date'] = pd.to_datetime(client_train.date).dt.date
# client_train['date'] = pd.to_datetime(client_train.date).dt.floor("D") # workaround for cudf

client_train['available_date'] = client_train['date'] + datetime.timedelta(days=2)

In [7]:
client_features = ['eic_count', 'installed_capacity']

## Weather features

In [8]:
from functools import reduce
from itertools import chain

def get_weighted_historical_weather_by_county(forecast_weather_df, time_column='datetime',shift_feature=True):

    weather_condition_columns = [
        'temperature',
        'dewpoint',
        'cloudcover_high',
        'cloudcover_low',
        'cloudcover_mid',
        'cloudcover_total',
        'direct_solar_radiation',
        'snowfall', 
        'rain',
    ]
    
    cumulative_features = [
        'direct_solar_radiation',
        'snowfall', 
        'rain',
    ]
    
    point_features = [
        'temperature',
        'dewpoint',
        'cloudcover_high',
        'cloudcover_low',
        'cloudcover_mid',
        'cloudcover_total',
    ]
    
    county_coordinate_weights_truncated = county_coordinate_weights.copy()
    
    new_df = forecast_weather_df.copy()
    
    max_time = new_df['datetime'].max()
    
    
    new_df[['latitude', 'longitude']] = new_df[['latitude', 'longitude']].astype(np.float64).round(1)
    
    # The forecast data served during the test phase may have duplicate/missing coordinate pairs
    # Need to reindex the dataframe to avoid errors
    coors = historical_weather_train[['latitude', 'longitude']].set_index(['latitude', 'longitude']).sort_index().index.unique()
    timestamps = new_df[time_column].unique()
    idx = [[ts] + [*coor] for ts in timestamps for coor in coors]
    reidx = pd.DataFrame(idx, columns=['datetime', 'latitude', 'longitude']).set_index(['datetime', 'latitude', 'longitude']).index
    
    weather_condition_arr = new_df.sort_values(
                            [time_column, 'latitude', 'longitude']
                            ).groupby(
                                [time_column, 'latitude', 'longitude']
                                )[weather_condition_columns].mean()
    
    weather_condition_arr = weather_condition_arr.reindex(reidx).values.reshape(-1, 112, len(weather_condition_columns))
    
    
    new_index = new_df.sort_values([time_column]).groupby(
                            [time_column]
                            )[weather_condition_columns].mean().index.drop_duplicates()


    new_index = np.repeat(new_index, 16)

    
    weather_values_arr = (np.swapaxes(weather_condition_arr, 1, 2)  @ county_coordinate_weights_truncated.set_index('county_id').values.T)
    weather_values_arr = np.swapaxes(weather_values_arr, 1, 2).reshape(-1, len(weather_condition_columns))
    
    
    weather_by_county = pd.DataFrame(weather_values_arr, columns=weather_condition_columns, index=new_index)
    weather_by_county['county'] = np.array(np.arange(16).tolist() * (weather_values_arr.shape[0]//16) )
    
    weather_by_county.reset_index(inplace=True)
    
    
    weather_by_county = weather_by_county.set_index(['county', 'datetime'])#.sort_index()
    
    weather_by_county = weather_by_county.reindex(
                            weather_by_county.index.tolist() + [(county, max_time + datetime.timedelta(hours=i)) for county in range(16) for i in range(1, 49)]
                        )
    
    weather_by_county = weather_by_county.sort_index()

    if shift_feature:
#         weather_by_county[cumulative_features] = weather_by_county[cumulative_features].shift(-1)
        weather_by_county[point_features] = (weather_by_county[point_features].shift(1) + weather_by_county[point_features]) / 2
        
    
    lag_weather_features = [
        'direct_solar_radiation', 
#         'temperature',
#         'cloudcover_low',
#         'dewpoint',
        ]
    
    for lag_day in [2, 7]:
        weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].shift(24*lag_day)
        weather_by_county[[f"{ft}_{lag_day}_day_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]].values 


    weather_by_county[[f"{ft}_7_day_mean" for ft in lag_weather_features]] = sum([weather_by_county[lag_weather_features].shift(24*i) for i in range(2, 9)]) / 7
    
    weather_by_county['temp_minus_dewpoint'] = weather_by_county['temperature'] - weather_by_county['dewpoint']
    
    weather_by_county = weather_by_county.reset_index()
    
    weather_by_county = weather_by_county.rename(columns={'temperature': 'temperature_h',
                                                         'dewpoint': 'dewpoint_h',
                                                         'direct_solar_radiation': 'direct_solar_radiation_h',
                                                          'direct_solar_radiation_lag_2_day': 'direct_solar_radiation_lag_2_day_h',
                                                          'direct_solar_radiation_lag_7_day': 'direct_solar_radiation_lag_7_day_h',
                                                          'direct_solar_radiation_7_day_mean': 'direct_solar_radiation_7_day_mean_h',
                                                          'temp_minus_dewpoint': 'temp_minus_dewpoint_h'
                                                         })
    return weather_by_county[['county', 'datetime', 'direct_solar_radiation_lag_2_day_h', 'direct_solar_radiation_lag_7_day_h', 'direct_solar_radiation_7_day_mean_h']]

In [9]:
historical_weather_train = pd.read_csv(comp_path + "/historical_weather.csv")
historical_weather_train['datetime'] = pd.to_datetime(historical_weather_train['datetime'])

In [10]:
forecast_weather_train = pd.read_csv(comp_path + "/forecast_weather.csv")
forecast_weather_train['forecast_datetime'] = pd.to_datetime(forecast_weather_train['forecast_datetime'])#.dt.tz_convert("EET").dt.tz_localize(None)

In [11]:
weather_features = [
    'temperature',
#     "dewpoint", 
#    "cloudcover_high", 
    "cloudcover_low", 
    "cloudcover_mid", 
    "cloudcover_total", 
    "10_metre_u_wind_component",
    "10_metre_v_wind_component",
#     "wind_speed",
    "direct_solar_radiation", 
    "surface_solar_radiation_downwards", 
#     "snowfall",
    'total_precipitation',
   ]


In [12]:
from functools import reduce
from itertools import chain

def get_weighted_forecast_by_county(forecast_weather_df, historical_weather_df,time_column='forecast_datetime',shift_feature=True):

    weather_condition_columns = [
        'temperature',
        'dewpoint',
        'cloudcover_high',
        'cloudcover_low',
        'cloudcover_mid',
        'cloudcover_total',
        '10_metre_u_wind_component',
        '10_metre_v_wind_component',
        'direct_solar_radiation',
        'surface_solar_radiation_downwards',
        'snowfall', 
        'total_precipitation',
    ]
    

    point_features= [
        'temperature', 
        'dewpoint', 
        'cloudcover_high',
        "cloudcover_low", 
        "cloudcover_mid", 
        "cloudcover_total", 
        "10_metre_u_wind_component",
        "10_metre_v_wind_component",
    ]

    cumulative_features= [
        'direct_solar_radiation', 
        'total_precipitation', 
        'surface_solar_radiation_downwards',
        'snowfall',
    ]

    new_df = forecast_weather_df.copy()
    new_df[['latitude', 'longitude']] = new_df[['latitude', 'longitude']].astype(np.float64).round(1)
    
    
    # The forecast data served during the test phase may have duplicate/missing coordinate pairs
    # Need to reindex the dataframe to avoid errors
    coors = historical_weather_train[['latitude', 'longitude']].set_index(['latitude', 'longitude']).sort_index().index.unique()
    timestamps = new_df[time_column].unique()
    
    idx = [[ts] + [*coor] for ts in timestamps for coor in coors]
#     idx = [[ts] + [*coor] for ts in timestamps.to_pandas() for coor in coors.to_pandas()]
    reidx = pd.DataFrame(idx, columns=['datetime', 'latitude', 'longitude']).set_index(['datetime', 'latitude', 'longitude']).index
    
    weather_condition_arr = new_df.sort_values(
                            [time_column, 'latitude', 'longitude']
                            ).groupby(
                                [time_column, 'latitude', 'longitude']
                                )[weather_condition_columns].mean()
    
    weather_condition_arr = weather_condition_arr.reindex(reidx).values.reshape(-1, 112, len(weather_condition_columns))
    
    
    new_index = new_df.sort_values([time_column]).groupby(
                            [time_column]
                            )[weather_condition_columns].mean().index.drop_duplicates()


    new_index = np.repeat(new_index, 16)
    
    
    weather_values_arr = (np.swapaxes(weather_condition_arr, 1, 2)  @ county_coordinate_weights.set_index('county_id').values.T)
    weather_values_arr = np.swapaxes(weather_values_arr, 1, 2).reshape(-1, len(weather_condition_columns))
    
    
    
    weather_by_county = pd.DataFrame(weather_values_arr, columns=weather_condition_columns, index=new_index)
    weather_by_county['county'] = np.array(np.arange(16).tolist() * (weather_values_arr.shape[0]//16) )
    
    weather_by_county.reset_index(inplace=True)
    weather_by_county['datetime'] = weather_by_county[time_column]
    
    
    
    weather_by_county = weather_by_county.set_index(['county', 'datetime']).sort_index()
    
    
    if shift_feature:
#         weather_by_county[cumulative_features] = weather_by_county[cumulative_features].shift(-1)
        weather_by_county[point_features] = (weather_by_county[point_features].shift(1) + weather_by_county[point_features]) / 2
        
    
    lag_weather_features = ['direct_solar_radiation', 
                            'temperature',
                            'cloudcover_low',
                            'dewpoint',
                            'total_precipitation',
                            'surface_solar_radiation_downwards']
    
    for lag_day in [2,3,7]:
        weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].shift(24*lag_day)
        weather_by_county[[f"{ft}_{lag_day}_day_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_lag_{lag_day}_day" for ft in lag_weather_features]].values 

    
    weather_by_county[[f"{ft}_7_day_mean" for ft in lag_weather_features]] = sum([weather_by_county[lag_weather_features].shift(24*i) for i in range(2, 9)]) / 7
    
    weather_by_county[[f"{ft}_7_day_mean_diff" for ft in lag_weather_features]] = weather_by_county[lag_weather_features].values  - weather_by_county[[f"{ft}_7_day_mean" for ft in lag_weather_features]].values 

    weather_by_county[['snowfall_1_day_rolling_sum']] = weather_by_county[['snowfall']].reset_index().groupby(['county'])[['snowfall']].rolling(24*1).sum().values

    weather_by_county['wind_speed'] = weather_by_county[['10_metre_u_wind_component', '10_metre_v_wind_component']].pow(2).sum(axis=1).pow(0.5)
    weather_by_county['temp_minus_dewpoint'] = weather_by_county['temperature'] - weather_by_county['dewpoint']
    
    weather_by_county = weather_by_county.reset_index()
    
    
    weather_by_county = weather_by_county.merge(
        historical_weather_df,
        how="left",
        on=["county", "datetime",],
        )
    
    # Override with historical weather
    weather_by_county[f"direct_solar_radiation_7_day_mean_diff"] = weather_by_county['direct_solar_radiation'].values  - weather_by_county['direct_solar_radiation_7_day_mean_h'].values 
    weather_by_county[f"direct_solar_radiation_2_day_diff"] = weather_by_county['direct_solar_radiation'].values  - weather_by_county['direct_solar_radiation_lag_2_day_h'].values 
    weather_by_county[f"direct_solar_radiation_7_day_diff"] = weather_by_county['direct_solar_radiation'].values  - weather_by_county['direct_solar_radiation_lag_7_day_h'].values 
    
    return weather_by_county

## Electricity & Gas Price

In [13]:
def process_elec_prices(process_elec_prices_df):
    
    new_df = process_elec_prices_df.copy()
    
    new_df.forecast_date = pd.to_datetime(new_df.forecast_date)
    new_df.origin_date = pd.to_datetime(new_df.origin_date)
    new_df['data_available_datetime'] = new_df.forecast_date + datetime.timedelta(days=1)
    
    
    new_df['data_available_date'] = new_df['data_available_datetime'].dt.date
#     new_df['data_available_date'] = new_df['data_available_datetime'].dt.floor("D")
    

    for i in range(1, 8):
        new_df[f'euros_per_mwh_lag_{i}_day'] = new_df['euros_per_mwh'].shift(24*i)

    euros_per_mwh_lag_7_day_mean = sum([new_df[f'euros_per_mwh_lag_{i}_day'] for i in range(1,8)]) / 7
    euros_per_mwh_lag_28_day_mean = sum([new_df['euros_per_mwh'].shift(24*i) for i in range(1,29)]) / 28

    new_df[f'euros_per_mwh_lag_7_day_mean_diff'] = new_df['euros_per_mwh'] - euros_per_mwh_lag_7_day_mean
    new_df[f'euros_per_mwh_lag_28_day_mean_diff'] = new_df['euros_per_mwh'] - euros_per_mwh_lag_28_day_mean

    new_df[f'euros_per_mwh_lag_7_day_diff'] = new_df['euros_per_mwh'] - new_df[f'euros_per_mwh_lag_7_day']
    new_df[f'euros_per_mwh_lag_2_day_diff'] = new_df['euros_per_mwh'] - new_df[f'euros_per_mwh_lag_2_day']


    daily_elec_prices = new_df.groupby(['data_available_date'])[['euros_per_mwh']].mean().reset_index()
    daily_elec_prices = daily_elec_prices.rename(columns={"euros_per_mwh": "daily_euros_per_mwh"})


    daily_elec_prices['daily_euros_per_mwh_lag_1_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(1)
    # daily_elec_prices['daily_euros_per_mwh_lag_2_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(2)
    # daily_elec_prices['daily_euros_per_mwh_lag_3_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(3)
    daily_elec_prices['daily_euros_per_mwh_lag_7_day'] = daily_elec_prices['daily_euros_per_mwh'].shift(7)
    daily_elec_prices['daily_euros_per_mwh_rolling_7_day_mean'] =  daily_elec_prices['daily_euros_per_mwh'].rolling(7).mean()
    
    
    new_df = new_df.merge(
                        daily_elec_prices,
                        on=['data_available_date'],
                        how='left'
                    )
    
    return new_df

In [14]:
def propcess_gas_prices(gas_prices_df):

    new_df = gas_prices_df.copy()
    new_df['forecast_date'] = pd.to_datetime(new_df['forecast_date']).dt.date
    new_df['data_available_date'] = new_df.forecast_date + datetime.timedelta(days=1)
    new_df['gas_mid_price'] = (new_df['lowest_price_per_mwh'] + new_df['highest_price_per_mwh']) /2
    
    new_df['gas_mid_price_lag_1_day'] = new_df['gas_mid_price'].shift(1)

    new_df['gas_mid_price_lag_7_day'] = new_df['gas_mid_price'].shift(7)
    new_df['gas_mid_price_rolling_7_day_mean'] =  new_df['gas_mid_price'].rolling(7).mean()
    
    return new_df
    

## Merge dataframe

In [15]:
%%time

train = train.merge(
    process_elec_prices(elec_prices)[[
        "euros_per_mwh", 
#          "daily_euros_per_mwh",
#          'daily_euros_per_mwh_lag_1_day',
#          'daily_euros_per_mwh_lag_7_day',
#          'daily_euros_per_mwh_rolling_7_day_mean',
        "data_available_datetime",
        "euros_per_mwh_lag_7_day_mean_diff",
        "euros_per_mwh_lag_28_day_mean_diff",
        "euros_per_mwh_lag_7_day_diff",
        "euros_per_mwh_lag_2_day_diff",
        ]],
    how="left",
    right_on="data_available_datetime",
    left_on="datetime",
   )

# train = train.merge(
#     propcess_gas_prices(gas_prices_train)[[
#         "data_available_date",
#         "gas_mid_price",
#         'gas_mid_price_lag_1_day',
#         'gas_mid_price_lag_7_day',
#         'gas_mid_price_rolling_7_day_mean',
#         ]],
#     how="left",
#     right_on="data_available_date",
#     left_on="date",
#    )



# train = train.merge(
#     client_train.drop("date", axis=1), 
#     on=["county", "is_business", "product_type", "data_block_id"], 
#     how="left")

train = train.merge(
        client_train.drop(["data_block_id", "date"], axis=1),  
        left_on=["county", "is_business", "product_type", "date"], 
        right_on=["county", "is_business", "product_type", "available_date"], 
        how="left")

train = train.merge(
    get_weighted_forecast_by_county(
        forecast_weather_train, 
        get_weighted_historical_weather_by_county(historical_weather_train)),
    how="left",
    on=["county", "datetime",],
    )


CPU times: user 26.6 s, sys: 2.76 s, total: 29.4 s
Wall time: 29.4 s


In [16]:
def fill_missing_time(df, start_time=None, end_time=None, inference=False):
    
    if start_time is None:
        start_time = df['datetime'].min()
        
    if end_time is None:
        end_time = df['datetime'].max()
        
    df_array = np.empty((16, 2, 4, 2), dtype=object)
    idx = pd.date_range(start_time, end_time, freq='H')
    
    for county in range(16):
        for is_business in [0, 1]:
            for product_type in [0, 1, 2, 3]:
                for is_consumption in [0, 1]:

#                     print(county, is_business, product_type, is_consumption, end="")
                    try:
                        temp_df = df[
                                    (df['county']==county) *
                                    (df['is_business']==is_business) *
                                    (df['product_type']==product_type) *
                                    (df['is_consumption']==is_consumption)
                                    ].set_index('datetime')
                        length = len(temp_df)
                
                        if inference and length==0:
                            continue

                        temp_df = temp_df.reindex(
                            idx, 
#                             method='nearest',
                        )
                        temp_df[["county", "is_business", "product_type", "is_consumption"]] = [county, is_business, product_type, is_consumption]
                        df_array[county, is_business, product_type, is_consumption] = temp_df

                            
#                         print(len(df_array[county, is_business, product_type, is_consumption]))
                    except:
#                         print(0)
                        pass

                for is_consumption in [0, 1]:
                    if df_array[county, is_business, product_type, 1-is_consumption] is None:
                        continue
                    df_array[county, is_business, product_type, is_consumption]['consumption_counterpart'] = df_array[county, is_business, product_type, 1-is_consumption]['target']
            
                
    new_df = pd.concat([e for e in df_array.flatten() if e is not None])
    new_df.index.names = ['datetime']
    new_df = new_df.reset_index()
    
    new_df['date'] = new_df['datetime'].dt.date 
#     new_df = new_df.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)
    return new_df, df_array

In [17]:
%%time
train, df_arr = fill_missing_time(train)

CPU times: user 22.7 s, sys: 4.71 s, total: 27.4 s
Wall time: 27.5 s


## Append Lag features

In [18]:
train = train.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)

base_lag = 48


grouped_eic_count = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['eic_count']
# train['eic_count_2_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*0).fillna(0)
# train['eic_count_7_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*5).fillna(0)
# train['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0).fillna(0) - np.mean([grouped_eic_count.shift(24*i).fillna(0) for i in range(0, 7)], axis=0)

train['eic_count_2_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*0)
train['eic_count_7_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*5)
train['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0) - np.mean([grouped_eic_count.shift(24*i) for i in range(0, 7)], axis=0)

grouped_installed_capacity = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['installed_capacity']
# train['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*0).fillna(0)
# train['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*5).fillna(0)
# train['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0).fillna(0) - np.mean([grouped_installed_capacity.shift(24*i).fillna(0) for i in range(0, 7)], axis=0)

train['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*0)
train['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*5)
train['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0) - np.mean([grouped_installed_capacity.shift(24*i) for i in range(0, 7)], axis=0)


### scale target by installed_capacity
# train['target_original'] = train['target']
# train['target'] = train['target'] / train['installed_capacity']
###


grouped_target = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target']

for i in range(2, 17):
    train[f'lag_{i}_day'] = grouped_target.shift(24*i).fillna(0)


train['lag_7_day_mean'] = np.mean([train[f'lag_{i}_day'] for i in range(2,9)], axis=0)
train['lag_8_14_day_mean'] = np.mean([train[f'lag_{i}_day'] for i in range(9,16)], axis=0)

train['lag_2_day_1h'] = grouped_target.shift(48+1)
train['lag_3_day_1h'] = grouped_target.shift(72+1)
train['lag_4_day_1h'] = grouped_target.shift(96+1)
train['lag_7_day_1h'] = grouped_target.shift(24*7+1)


train['rolling_mean_24'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).mean().shift(base_lag).values

train['rolling_mean_24x7x4'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24*7*4).mean().shift(base_lag).values
train['rolling_std_24'] = train.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).std().shift(base_lag).values


grouped_consumption_counterpart = train.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart']

for i in range(2, 9):
    train[f'lag_{i}_day_consumption_counterpart'] = grouped_target.shift(24*i)
train[f'lag_2_day_1h_consumption_counterpart'] = grouped_target.shift(24*2+1)

train['lag_7_day_mean_consumption_counterpart'] = np.mean([train[f'lag_{i}_day_consumption_counterpart'] for i in range(2,9)], axis=0)

train = train.reset_index()

In [19]:
# train = train[train['datetime'] < pd.to_datetime("2022-09-01 00:00:00")]


In [20]:
national_holidays =[
                    [2021, 1, 1],
                    [2021, 2, 24],
                    [2021, 4, 2],
                    [2021, 4, 4],
                    [2021, 5, 1],
                    [2021, 5, 23],
                    [2021, 6, 23],
                    [2021, 6, 24],
                    [2021, 8, 20],
                    [2021, 12, 24],
                    [2021, 12, 25],
                    [2021, 12, 26],


                    [2022, 1, 1],
                    [2022, 2, 24],
                    [2022, 4, 15],
                    [2022, 4, 17],
                    [2022, 5, 1],
                    [2022, 6, 5],
                    [2022, 6, 23],
                    [2022, 6, 24],
                    [2022, 8, 20],
                    [2022, 12, 24],
                    [2022, 12, 25],
                    [2022, 12, 26],

                    [2023, 1, 1],
                    [2023, 2, 24],
                    [2023, 4, 7],
                    [2023, 4, 9],
                    [2023, 5, 1],
                    [2023, 5, 28],
                    [2023, 6, 23],
                    [2023, 6, 24],
                    [2023, 8, 20],
                    [2023, 12, 24],
                    [2023, 12, 25],
                    [2023, 12, 26],

                    [2024, 1, 1],
                    [2024, 2, 24],
                    [2024, 3, 29],
                    [2024, 3, 31],
                    [2024, 5, 1],
                    [2024, 5, 19],
                    [2024, 6, 23],
                    [2024, 6, 24],
                    [2024, 8, 20],
                    [2024, 12, 24],
                    [2024, 12, 25],
                    [2024, 12, 26],
]

In [21]:
import datetime
national_holidays_dt = [pd.to_datetime('-'.join(str(n) for n in holiday)).date() for holiday in national_holidays]

def append_daytype(train_df):
    train_df['days_in_month'] = train_df['datetime'].dt.days_in_month

    train_df['inner_weekday'] = (train_df['datetime'].dt.dayofweek >= 1) * (train_df['datetime'].dt.dayofweek <= 3) # Tuesday to Thursday
    train_df['holiday'] = (train_df['datetime'].dt.dayofweek >= 5) * (train_df['datetime'].dt.dayofweek <= 6) # Saturday and Sunday
    train_df['pre_holiday'] = train_df['datetime'].dt.dayofweek == 4
    train_df['post_holiday'] = train_df['datetime'].dt.dayofweek == 0
    
    train_df['national_holiday'] = train_df['datetime'].dt.date.isin(national_holidays_dt)
    train_df['pre_national_holiday'] = train_df['datetime'].dt.date.isin(d-datetime.timedelta(days=1) for d in national_holidays_dt)
    train_df['post_national_holiday'] = train_df['datetime'].dt.date.isin(d+datetime.timedelta(days=1) for d in national_holidays_dt)
    
    train_df['daytype'] = None
    train_df.loc[train_df['inner_weekday'], 'daytype'] = 'inner_weekday'
    train_df.loc[train_df['pre_holiday'] + train_df['pre_national_holiday'], 'daytype'] = 'pre_holiday'
    train_df.loc[train_df['post_holiday'] + train_df['post_national_holiday'], 'daytype'] = 'post_holiday'
    train_df.loc[train_df['holiday'] + train_df['national_holiday'], 'daytype'] = 'holiday'
    

    
    train_df['daytype'] = train_df['daytype'].astype(dtype)
    train_df['daytype_lag_2_day'] = train_df['daytype'].shift(24*2)
    train_df['daytype_lag_3_day'] = train_df['daytype'].shift(24*3)
    train_df['daytype_lag_7_day'] = train_df['daytype'].shift(24*7)
    
    fts_to_remove = ['inner_weekday', 'holiday', 'pre_holiday', 'post_holiday', 'national_holiday', 'pre_national_holiday', 'post_national_holiday']
    for ft in fts_to_remove:
        del train_df[ft]

    
    return train_df

In [22]:
def append_school_holiday(train_df):
    """
    school holidays:
    
    Feb 22 - Feb 28,
    Apr 19 - Apr 25,
    Jun 14 - Aug 31,
    Oct 25 - Oct 31,
    Dec 23, 2021 - Jan 9, 2022,

    Feb 28 - Mar 6,
    Apr 25 - May 1,
    Jun 14 - Aug 31,
    Oct 24 - Oct 30,
    Dec 22, 2022 - Jan 8, 2023,

    Feb 27 - Mar 5,
    Apr 24 - Apr 30,
    Jun 14 - Aug 31,
    Oct 23 - Oct 29,
    Dec 21, 2023 - Jan 7, 2024,
    """
    
    school_holiday_range = [
        ["2021-02-22", "2021-02-28"],
        ["2021-04-19", "2021-04-25"],
        ["2021-06-14", "2021-08-31"],
        ["2021-10-25", "2021-10-31"],
        ["2021-12-23", "2022-01-09"],

        ["2022-02-23", "2022-03-06"],
        ["2022-04-25", "2022-05-01"],
        ["2022-06-14", "2022-08-31"],
        ["2022-10-24", "2022-10-30"],    
        ["2022-12-22", "2023-01-08"],

        ["2023-02-27", "2023-03-05"],
        ["2023-04-24", "2023-04-30"],
        ["2023-06-14", "2023-08-31"],
        ["2023-10-23", "2023-10-29"],    
        ["2023-12-21", "2024-01-07"],

        ["2024-02-26", "2023-03-03"],
        ["2023-04-22", "2023-04-28"],
        ["2023-06-13", "2023-08-31"],]

    school_holiday_masks = [(train_df['datetime'] >= pd.to_datetime(start_date)) * (train_df['datetime'] <= pd.to_datetime(end_date)) 
                            for start_date, end_date in school_holiday_range]
    
    train_df['school_holiday'] = np.sum(school_holiday_masks,axis=0).astype(bool)
    
    return train_df

In [23]:
%%time

# dtype = str
dtype = "category"


train['dayofweek'] = train['datetime'].dt.dayofweek.astype(dtype)
train["month"]  = train["datetime"].dt.month
train['day'] = train['datetime'].dt.day
train['yearmonth'] = train['datetime'].dt.year.astype(str) + train['datetime'].dt.month.astype(str)

train = append_daytype(train)
train = append_school_holiday(train)

hours_in_day = 24
train['hour'] = train['datetime'].dt.hour
train['sin_hour'] = np.sin(2*np.pi*train.hour/hours_in_day)
train['cos_hour'] = np.cos(2*np.pi*train.hour/hours_in_day)
train['hour'] = train['hour'].astype(dtype)


months_in_year = 12
train['sin_month'] = np.sin(2*np.pi*train.month/months_in_year)
train['cos_month'] = np.cos(2*np.pi*train.month/months_in_year)


train['county'] = train['county'].astype(dtype)
train['product_type'] = train['product_type'].astype(dtype)

CPU times: user 20.1 s, sys: 1.38 s, total: 21.4 s
Wall time: 21.4 s


In [24]:
train = train[~train['target'].isna()]

print(int(100*len(train)/2018352),"% of dataset")
train.head()

99 % of dataset


      county  is_business product_type  is_consumption            datetime  \
30624      0            0            1               0 2021-09-01 00:00:00   
30625      0            0            1               0 2021-09-01 01:00:00   
30626      0            0            1               0 2021-09-01 02:00:00   
30627      0            0            1               0 2021-09-01 03:00:00   
30628      0            0            1               0 2021-09-01 04:00:00   

       target  data_block_id  row_id  prediction_unit_id        date  ...  \
30624   0.713            0.0     0.0                 0.0  2021-09-01  ...   
30625   1.132            0.0   122.0                 0.0  2021-09-01  ...   
30626   0.490            0.0   244.0                 0.0  2021-09-01  ...   
30627   0.496            0.0   366.0                 0.0  2021-09-01  ...   
30628   0.149            0.0   488.0                 0.0  2021-09-01  ...   

             daytype daytype_lag_2_day  daytype_lag_3_day  daytype_l

In [25]:
raw_features = [
                "county",
                "is_business",
                "product_type",
                "is_consumption",
                "dayofweek",
                "daytype",
#                 "sin_month",
#                 "cos_month",
#                 "hour",
                "sin_hour",
                "cos_hour",
                "lag_2_day",
                "lag_3_day",
                "lag_4_day",
                "lag_7_day_mean",
                "lag_5_day",
                "lag_6_day",
                "lag_7_day",
                "lag_2_day_1h",
#                 "lag_3_day_1h",
#                 "lag_4_day_1h",
#                 "lag_7_day_1h",
                "rolling_mean_24",
#                 "rolling_mean_24x7",
                "rolling_std_24",
                "lag_2_day_consumption_counterpart",
#                 "lag_3_day_consumption_counterpart",
#                 "lag_4_day_consumption_counterpart",
#                 "lag_5_day_consumption_counterpart",
#                 "lag_6_day_consumption_counterpart",
#                 "lag_7_day_consumption_counterpart",
                "lag_7_day_mean_consumption_counterpart",
#                 "euros_per_mwh",
#                 "daily_euros_per_mwh", 
#                 'daily_euros_per_mwh_lag_1_day',
#                 'daily_euros_per_mwh_lag_7_day',
#                 'daily_euros_per_mwh_rolling_7_day_mean',
                # Gas price
#                 "gas_mid_price",
#                 'gas_mid_price_lag_1_day',
#                 'gas_mid_price_lag_7_day',
#                 'gas_mid_price_rolling_7_day_mean',
                'snowfall_1_day_rolling_sum',
#                 "school_holiday",
#                 'wind_speed',
                'eic_count_7_day_mean_diff',
                'installed_capacity_7_day_mean_diff',
               ] # + [f'product_type_embed_{i}' for i in range(8)] #+ [f'county_embed_{i}' for i in range(2)]


In [26]:
train.reset_index(drop=True, inplace=True)
# X = train.loc[train[train['hour'] == 0].index][features]
# Y = np.array([train['target'][train[train['hour'] == 0].index + i].values for i in range(24)]).T
# X.shape, Y.shape

In [27]:
weights =  1 / (train['month'].value_counts() / np.mean(train['month'].value_counts()))
weights = weights.reset_index().rename(columns={"count":"weight"})
weights['weight'] = weights['weight'].pow(0.5)

train = train.merge(
            weights, 
            on=['month'], 
            how='left'
        )

In [28]:
def select_features(raw_features, weather_features, client_features, is_consumption, target='target', verbose=0):
    
    features = raw_features + weather_features + client_features
    features.remove('is_consumption')
#     features.remove('is_business')
    
    fts_to_remove = []
    fts_to_add = []
    
    
    if is_consumption==0:
        fts_to_remove += [
            "lag_4_day",
            "lag_5_day",
            "lag_6_day",
#             "lag_7_day",
            
#             "lag_2_day_consumption_counterpart",
#             "lag_7_day_consumption_counterpart",
        ]
    elif is_consumption==1:
        fts_to_remove += [
#             'cloudcover_low',
            'cloudcover_mid',
            'cloudcover_total',
            '10_metre_u_wind_component',
            '10_metre_v_wind_component',
#             "wind_speed",
#             "dewpoint",
            "lag_5_day",
            "lag_6_day",
#             "sin_hour"
#             "total_precipitation",
            "sin_hour",
            "cos_hour",
        ]
        
        fts_to_add += [
            'hour'
        ]
        
    else:
        raise ValueError
        
        
    if is_consumption==0:
        fts_to_add += [
            'temp_minus_dewpoint',
            
            'direct_solar_radiation_7_day_mean_diff', 
#             'temperature_7_day_mean_diff', 
            'cloudcover_low_7_day_mean_diff',
#             'dewpoint_7_day_mean_diff',
            'total_precipitation_7_day_mean_diff',
            'surface_solar_radiation_downwards_7_day_mean_diff',
            
            
            'direct_solar_radiation_2_day_diff', 
#             'temperature_2_day_diff', 
            'cloudcover_low_2_day_diff', 
#             'dewpoint_2_day_diff', 
            'total_precipitation_2_day_diff', 
            'surface_solar_radiation_downwards_2_day_diff',
            
#             'direct_solar_radiation_3_day_diff', 
#             'temperature_3_day_diff', 
#             'cloudcover_low_3_day_diff', 
#             'dewpoint_3_day_diff', 
#             'total_precipitation_3_day_diff', 
#             'surface_solar_radiation_downwards_3_day_diff',
            
#             'wind_speed',
#             'daily_euros_per_mwh_lag_1_day',
#             'daily_euros_per_mwh_lag_7_day', 
#             'daily_euros_per_mwh_rolling_7_day_mean',
                
                # Gas price
#                 'gas_mid_price_lag_1_day',
#                 'gas_mid_price_lag_7_day',
#                 'gas_mid_price_rolling_7_day_mean',
#             'euros_per_mwh',
#             "lag_8_14_day_mean",
        ]
    elif is_consumption==1:
        fts_to_add += [
            'euros_per_mwh',
            'euros_per_mwh_lag_7_day_mean_diff',
#             'euros_per_mwh_lag_28_day_mean_diff',
#             'euros_per_mwh_lag_7_day_diff',
#             'euros_per_mwh_lag_2_day_diff',
            "lag_8_14_day_mean",
#             'daily_euros_per_mwh',
#             "lag_7_day_1h",
#             'direct_solar_radiation_7_day_mean_diff', 
#             'cloudcover_low_7_day_mean_diff',
#             'dewpoint_7_day_mean_diff',
#             'total_precipitation_7_day_mean_diff',
#             'surface_solar_radiation_downwards_7_day_mean_diff',
#             "daytype_lag_2_day",
#             'direct_solar_radiation_7_day_diff', 
#             "daytype_lag_7_day",
#             'lag_2_day_1h_consumption_counterpart',
#             "rolling_mean_24x7x4",
#             "school_holiday",
#                 'eic_count_7_day_mean_diff',
#                 'installed_capacity_7_day_mean_diff',
        ]
    else:
        raise ValueError

        
    if target in ['lag_2_day_diff', 'lag_2_day_diff_neg']:
        fts_to_add += [
            'daytype_lag_2_day',
            
            'temperature_2_day_diff', 
            'direct_solar_radiation_2_day_diff', 
            'surface_solar_radiation_downwards_2_day_diff',
            'total_precipitation_2_day_diff', 
            
            
            'euros_per_mwh_lag_2_day_diff',
#             'eic_count_2_day_diff',
#             'installed_capacity_2_day_diff',
        ]
        
        fts_to_remove += [
            "euros_per_mwh_lag_7_day_mean_diff",
        ]
        
    elif target in ['lag_7_day_diff', 'lag_7_day_diff_neg']:
        fts_to_add += [
            'daytype_lag_7_day',
            
            "lag_7_day_consumption_counterpart",
            'temperature_7_day_diff', 
            'direct_solar_radiation_7_day_diff', 
            'surface_solar_radiation_downwards_7_day_diff',
            'total_precipitation_7_day_diff', 
            
            
            'euros_per_mwh_lag_7_day_diff',
            'eic_count_7_day_diff',
            'installed_capacity_7_day_diff'
        ]
        
        fts_to_remove += [
            "euros_per_mwh_lag_7_day_mean_diff",
        ]
        

    features += fts_to_add
        
        
    for ft in fts_to_remove:
        try:
            features.remove(ft)
        except:
            print(f'Removing aborted: {ft} not found')
        
    if verbose==1:
        print(f'removing features: {fts_to_remove}')
        print(f'Adding features: {fts_to_add}')
        print(f'Final features: ({len(features)}) {features}')
        
    return list(set(features))

In [29]:
import re

In [30]:
def get_monotone_constraints_map(features, is_consumption, is_business=0):
    
    monotone_constraints_map = {ft: 0 for ft in features}
    

    if is_consumption==0:
#         monotone_constraints_map['euros_per_mwh'] = 1
#         monotone_constraints_map['daily_euros_per_mwh'] = 1
#         monotone_constraints_map['wind_speed'] = 1
    
        monotone_constraints_map['installed_capacity'] = 1
        monotone_constraints_map["direct_solar_radiation"] = 1
        monotone_constraints_map["cloudcover_total"] = -1
        monotone_constraints_map["cloudcover_mid"] = -1
        monotone_constraints_map["cloudcover_low"] = -1
        
        
#         monotone_constraints_map['cloudcover_low_2_day_diff'] = -1
#         monotone_constraints_map['direct_solar_radiation_2_day_diff'] = 1
#         monotone_constraints_map['surface_solar_radiation_downwards_2_day_diff'] = 1

#         monotone_constraints_map['cloudcover_low_7_day_mean_diff'] = -1
#         monotone_constraints_map['direct_solar_radiation_7_day_mean_diff'] = 1
#         monotone_constraints_map['surface_solar_radiation_downwards_7_day_mean_diff'] = 1



#         for ft in features:
#             if re.search('direct_solar_radiation_[0-9]_day_diff', ft):
#                 monotone_constraints_map[ft] = 1
#             if re.search('surface_solar_radiation_downwards_[0-9]_day_diff', ft):
#                 monotone_constraints_map[ft] = 1
#             if re.search('euros_per_mwh_lag_[0-9]_day_diff', ft):
#                 monotone_constraints_map[ft] = 1
                
    elif is_consumption==1:
#         monotone_constraints_map['euros_per_mwh'] = -1
#         monotone_constraints_map['daily_euros_per_mwh'] = -1
        monotone_constraints_map['eic_count'] = 1
    
    else:
        raise valueError

#         monotone_constraints_map['direct_solar_radiation_7_day_mean_diff'] = 1



        
    return monotone_constraints_map

In [31]:
# target_by_county = \
#     train.groupby([ 'dayofweek', 'hour', 'county', 'product_type','is_consumption'])['target'].sum().values.reshape((-1, 16, 4, 2)) \
# / (train.groupby(['county', 'product_type', 'is_consumption'])['target'].sum().values.reshape((16, 4, 2)) + 0.1)

# target_by_county = np.swapaxes(target_by_county, 0, 1).reshape(16, 4, -1)

In [32]:
# # target_by_county = np.swapaxes(target_by_county, 0, 1).reshape(4, -1)
# target_by_county =target_by_county.reshape(64, 336)
# target_by_county.shape

In [33]:
# from sklearn.decomposition import TruncatedSVD
# from scipy.sparse import csr_matrix

# svd = TruncatedSVD(n_components=8, n_iter=7, random_state=42)
# svd.fit(target_by_county)

# print(svd.explained_variance_ratio_, sum(svd.explained_variance_ratio_))
# # svd.transform(target_by_county)

In [34]:
# for i in range(8):
#     try:
#         del train[f'product_type_embed_{i}']
#     except:
#         pass
        
# county_embed = pd.DataFrame(svd.transform(target_by_county), columns=[f'product_type_embed_{i}' for i in range(8)]).reset_index()
# # county_embed = county_embed.rename(columns={'index': 'product_type'})


# county_embed['county'] = county_embed['index'] // 4
# county_embed['product_type'] = county_embed['index'] % 4
# train = train.merge(
#     county_embed, 
#     on=["county", "product_type"], 
#     how="left")

In [35]:
train['lag_7_day_mean_diff'] = np.clip(train['target'] - train['lag_7_day_mean'], 0, np.inf)
train['lag_7_day_mean_diff_neg'] = np.clip(train['lag_7_day_mean'] - train['target'], 0, np.inf)


train['lag_2_day_diff'] = np.clip(train['target'] - train['lag_2_day'], 0, np.inf)
train['lag_2_day_diff_neg'] = np.clip(train['lag_2_day'] - train['target'], 0, np.inf)

train['lag_7_day_diff'] = np.clip(train['target'] - train['lag_7_day'], 0, np.inf)
train['lag_7_day_diff_neg'] = np.clip(train['lag_7_day'] - train['target'], 0, np.inf)


In [36]:
train_augmented = train[-200000:].copy()

aug_factor = np.random.randn(200000) * 0.1 + 0.2 + 1

for target in [
    'installed_capacity',
    'eic_count',
        'target', 
       'lag_7_day_mean_diff', 
       'lag_7_day_mean_diff_neg',
       'lag_2_day_diff', 
       'lag_2_day_diff_neg',
       'lag_7_day_diff',
       'lag_7_day_diff_neg',
    
    'lag_2_day_consumption_counterpart', 
    'lag_7_day_mean_consumption_counterpart', 
    'eic_count_7_day_mean_diff', 
    'installed_capacity_7_day_mean_diff',
      ]:
    train_augmented[target] = train_augmented[target] * aug_factor

In [37]:
# for target in [
#         'target', 
#        'lag_7_day_mean_diff', 
#        'lag_7_day_mean_diff_neg',
#        'lag_2_day_diff', 
#        'lag_2_day_diff_neg',
#        'lag_7_day_diff',
#        'lag_7_day_diff_neg'
#       ]:
#     train['scaled_'+target] = train[target] / train['installed_capacity'] * 1000


In [38]:
import gc
try:
    for i in range(8,16):
        del train[f'lag_{i}_day'] 
except:
    pass
gc.collect()

0

In [39]:
# TODO match the lag features with the target model

In [40]:
# train.set_index(['county','is_business','product_type','is_consumption','datetime']).sort_index()[60:80][['eic_count', 'eic_count_2_day_diff', 'eic_count_7_day_diff', 'eic_count_7_day_mean_diff']]

# Train Models

In [41]:
train_augmented = pd.concat([train, train_augmented])

In [42]:
%%time

k_fold = 5

gkf = GroupKFold(n_splits=k_fold, )

results = {}

for is_consumption in [0, 1]:
# for is_consumption in [1]:

    
    for target in [
        'target', 
       'lag_7_day_mean_diff', 
       'lag_7_day_mean_diff_neg',
       'lag_2_day_diff', 
       'lag_2_day_diff_neg',
       'lag_7_day_diff',
       'lag_7_day_diff_neg'
      ]:
        
#         target = 'scaled_' + target
        
        
        print(f" Garbage collected: {gc.collect()}")
#     for is_business in [0, 1]:
#         key = f"is_consumption={is_consumption}, is_business={is_business}"
        key = (is_consumption, target)
#         key = f"is_consumption={is_consumption}"
    
        print(f"key: {key}")
        
        train_subset = train_augmented[
            (train['is_consumption'] == is_consumption)
        ]
        
#         train_subset = train[
#             (train['is_consumption'] == is_consumption)
#         ]
        
        
        train_subset.reset_index(drop=True, inplace=True)

        features = select_features(raw_features, weather_features, client_features, is_consumption=is_consumption, target=target, verbose=1)

        X = train_subset[features]
        Y = train_subset[target]

        G = train_subset['month']
#         G = train_subset['day']
        
        
        folds = gkf.split(X, Y, G)
        
#         tscv = TimeSeriesSplit(n_splits=5, )        
#         folds = tscv.split(X, Y, G)
        


        # categories = ['county','product_type', 'dayofweek']
        # categories = ['county','product_type', 'daytype']
        categories = ['county','product_type', 'daytype', 'dayofweek']
#         categories = ['county','product_type', 'daytype', 'dayofweek', 'school_holiday']
#         categories = ['daytype', 'dayofweek']
#         categories = ['product_type', 'daytype', 'dayofweek']
        if is_consumption:
            categories += ['hour']


        lgb_train = lgb.Dataset(X, 
                                Y, 
                               )
    
        params = {
            'boosting_type': 'gbdt',
            'objective': 'tweedie',
#             'objective': 'l1',
            'metric': 'mae', 
            'learning_rate': 0.1,
            'feature_fraction': 0.9, 
            'cat_l2': 0.1,
    #         'reg_sqrt': True,
            'max_depth': 7,
            'num_leaves': 127,
            'monotone_constraints': [value*((key[-3:]!='neg')*2-1) for key, value in get_monotone_constraints_map(features, is_consumption).items()],
            'monotone_constraints_method': 'advanced',
            'random_state': 42,
    #         'monotone_penalty': 0.01,
#             'bagging_fraction': 0.8,
#             'bagging_freq': 2,
            'reg_alpha': 0.2, 
            'reg_lambda': 0.2, 
        #     'extra_trees': True,
            'verbose': -1,
#             'device_type': 'cuda' 
        }
    
        if is_consumption==0:
            params['tweedie_variance_power'] = 1.4
        elif is_consumption==1:
            params['tweedie_variance_power'] = 1.3
        

        results[key] = lgb.cv(
            params,
            lgb_train,
            categorical_feature=categories, 
            folds=folds,
            num_boost_round=1000,
#             num_boost_round=50,
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=True),
                lgb.log_evaluation(50)
            ],
            return_cvbooster=True
        )


 Garbage collected: 0
key: (0, 'target')


<timed exec>:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


removing features: ['lag_4_day', 'lag_5_day', 'lag_6_day']
Adding features: ['temp_minus_dewpoint', 'direct_solar_radiation_7_day_mean_diff', 'cloudcover_low_7_day_mean_diff', 'total_precipitation_7_day_mean_diff', 'surface_solar_radiation_downwards_7_day_mean_diff', 'direct_solar_radiation_2_day_diff', 'cloudcover_low_2_day_diff', 'total_precipitation_2_day_diff', 'surface_solar_radiation_downwards_2_day_diff']
Final features: (39) ['county', 'is_business', 'product_type', 'dayofweek', 'daytype', 'sin_hour', 'cos_hour', 'lag_2_day', 'lag_3_day', 'lag_7_day_mean', 'lag_7_day', 'lag_2_day_1h', 'rolling_mean_24', 'rolling_std_24', 'lag_2_day_consumption_counterpart', 'lag_7_day_mean_consumption_counterpart', 'snowfall_1_day_rolling_sum', 'eic_count_7_day_mean_diff', 'installed_capacity_7_day_mean_diff', 'temperature', 'cloudcover_low', 'cloudcover_mid', 'cloudcover_total', '10_metre_u_wind_component', '10_metre_v_wind_component', 'direct_solar_radiation', 'surface_solar_radiation_downwar

/opt/conda/lib/python3.10/site-packages/lightgbm/basic.py:2068: UserWarning: categorical_feature in Dataset is overridden.
New categorical_feature is ['county', 'dayofweek', 'daytype', 'product_type']
  _log_warning('categorical_feature in Dataset is overridden.\n'


Training until validation scores don't improve for 100 rounds
[50]	cv_agg's l1: 24.3043 + 14.3592
[100]	cv_agg's l1: 23.3653 + 13.8079
[150]	cv_agg's l1: 23.078 + 13.681
[200]	cv_agg's l1: 23.0021 + 13.6858
[250]	cv_agg's l1: 22.9392 + 13.692
[300]	cv_agg's l1: 22.8847 + 13.6684
[350]	cv_agg's l1: 22.8639 + 13.6852
[400]	cv_agg's l1: 22.8308 + 13.6748
[450]	cv_agg's l1: 22.8074 + 13.6506
[500]	cv_agg's l1: 22.8094 + 13.6503
[550]	cv_agg's l1: 22.8315 + 13.6668
Early stopping, best iteration is:
[469]	cv_agg's l1: 22.784 + 13.6256
 Garbage collected: 549
key: (0, 'lag_7_day_mean_diff')


...

Training until validation scores don't improve for 100 rounds
[50]	cv_agg's l1: 35.4315 + 7.68876
[100]	cv_agg's l1: 33.5406 + 7.6295
[150]	cv_agg's l1: 32.6406 + 7.73478
[200]	cv_agg's l1: 32.1877 + 7.78706
[250]	cv_agg's l1: 31.9464 + 7.85492
[300]	cv_agg's l1: 31.7153 + 7.83283
[350]	cv_agg's l1: 31.6086 + 7.857
[400]	cv_agg's l1: 31.4978 + 7.84462
[450]	cv_agg's l1: 31.4235 + 7.83439
[500]	cv_agg's l1: 31.3605 + 7.86446
[550]	cv_agg's l1: 31.4322 + 8.08486
Early stopping, best iteration is:
[497]	cv_agg's l1: 31.3203 + 7.79183
 Garbage collected: 426
key: (1, 'lag_7_day_diff_neg')


<timed exec>:32: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


removing features: ['cloudcover_mid', 'cloudcover_total', '10_metre_u_wind_component', '10_metre_v_wind_component', 'lag_5_day', 'lag_6_day', 'sin_hour', 'cos_hour', 'euros_per_mwh_lag_7_day_mean_diff']
Adding features: ['hour', 'euros_per_mwh', 'euros_per_mwh_lag_7_day_mean_diff', 'lag_8_14_day_mean', 'daytype_lag_7_day', 'lag_7_day_consumption_counterpart', 'temperature_7_day_diff', 'direct_solar_radiation_7_day_diff', 'surface_solar_radiation_downwards_7_day_diff', 'total_precipitation_7_day_diff', 'euros_per_mwh_lag_7_day_diff', 'eic_count_7_day_diff', 'installed_capacity_7_day_diff']
Final features: (37) ['county', 'is_business', 'product_type', 'dayofweek', 'daytype', 'lag_2_day', 'lag_3_day', 'lag_4_day', 'lag_7_day_mean', 'lag_7_day', 'lag_2_day_1h', 'rolling_mean_24', 'rolling_std_24', 'lag_2_day_consumption_counterpart', 'lag_7_day_mean_consumption_counterpart', 'snowfall_1_day_rolling_sum', 'eic_count_7_day_mean_diff', 'installed_capacity_7_day_mean_diff', 'temperature', 'cl

/opt/conda/lib/python3.10/site-packages/lightgbm/basic.py:2068: UserWarning: categorical_feature in Dataset is overridden.
New categorical_feature is ['county', 'dayofweek', 'daytype', 'hour', 'product_type']
  _log_warning('categorical_feature in Dataset is overridden.\n'


Training until validation scores don't improve for 100 rounds
[50]	cv_agg's l1: 29.0429 + 3.87041
[100]	cv_agg's l1: 27.6119 + 3.83781
[150]	cv_agg's l1: 27.0544 + 3.81593
[200]	cv_agg's l1: 26.7606 + 3.80156
[250]	cv_agg's l1: 26.5541 + 3.83593
[300]	cv_agg's l1: 26.4312 + 3.883
[350]	cv_agg's l1: 26.3216 + 3.92201
[400]	cv_agg's l1: 26.2134 + 3.94407
[450]	cv_agg's l1: 26.1325 + 3.94271
[500]	cv_agg's l1: 26.0677 + 3.93143
[550]	cv_agg's l1: 26.0622 + 3.93825
[600]	cv_agg's l1: 26.0286 + 3.95118
[650]	cv_agg's l1: 26.0173 + 3.96085
[700]	cv_agg's l1: 26.0162 + 3.95486
[750]	cv_agg's l1: 26.0147 + 3.98233
[800]	cv_agg's l1: 25.9981 + 3.99683
[850]	cv_agg's l1: 25.9814 + 3.98826
[900]	cv_agg's l1: 25.9669 + 3.99962
[950]	cv_agg's l1: 25.9739 + 4.00978
Early stopping, best iteration is:
[892]	cv_agg's l1: 25.9624 + 3.99761
CPU times: user 5h 22min 23s, sys: 8min 4s, total: 5h 30min 27s
Wall time: 1h 43min 37s


In [43]:
gc.collect()

4

In [44]:
list(results.keys())

[(0, 'target'),
 (0, 'lag_7_day_mean_diff'),
 (0, 'lag_7_day_mean_diff_neg'),
 (0, 'lag_2_day_diff'),
 (0, 'lag_2_day_diff_neg'),
 (0, 'lag_7_day_diff'),
 (0, 'lag_7_day_diff_neg'),
 (1, 'target'),
 (1, 'lag_7_day_mean_diff'),
 (1, 'lag_7_day_mean_diff_neg'),
 (1, 'lag_2_day_diff'),
 (1, 'lag_2_day_diff_neg'),
 (1, 'lag_7_day_diff'),
 (1, 'lag_7_day_diff_neg')]

In [45]:
display(
    pd.DataFrame(
     [(is_consumption, 
       target, 
       np.mean([results[(is_consumption, target)]['cvbooster'].boosters[i].eval_valid()[0][2] for i in range(k_fold)]),
       np.std([results[(is_consumption, target)]['cvbooster'].boosters[i].eval_valid()[0][2] for i in range(k_fold)]),
      )
      for is_consumption, target in list(results.keys())]
 )
)

    0                        1          2          3
0   0                   target  22.825259  13.664893
1   0      lag_7_day_mean_diff  13.315963   7.167026
2   0  lag_7_day_mean_diff_neg  10.450834   6.921003
3   0           lag_2_day_diff  13.906536   7.927591
4   0       lag_2_day_diff_neg  11.187784   7.979881
5   0           lag_7_day_diff  14.817464   7.851575
6   0       lag_7_day_diff_neg  10.689869   7.436208
7   1                   target  53.965071   5.151156
8   1      lag_7_day_mean_diff  33.306998   5.642659
9   1  lag_7_day_mean_diff_neg  23.325513   2.813375
10  1           lag_2_day_diff  29.461363   3.214756
11  1       lag_2_day_diff_neg  24.994476   2.132699
12  1           lag_7_day_diff  31.395517   8.119241
13  1       lag_7_day_diff_neg  25.982493   4.008172

In [46]:
def get_boosters(key, i=None):
    boosters = results[key]["cvbooster"].boosters
    
    if i is not None:
        boosters = [boosters[i]]
        
    return boosters

def get_prediciton(input_df_ori, results, is_consumptions=[0, 1], i=None, scaled=False):
    
    input_df = input_df_ori.copy()
    try:
        del input_df['pred']
    except:
        pass
    
    input_df['pred'] = None
    
    keys = list(results.keys())
    
    for is_consumption in is_consumptions:
        
        if is_consumption:
#             coef_0, coef_1, coef_2, coef_3 = 1/4, 1/4, 1/4, 1/4
            coef_0, coef_1, coef_2, coef_3 = 2/7, 2/7, 2/7, 1/7
        else:
#              coef_0, coef_1, coef_2, coef_3 = 1/4, 1/4, 1/4, 1/4
            coef_0, coef_1, coef_2, coef_3 = 2/7, 2/7, 2/7, 1/7
    

        input_df_subset = input_df[
            (input_df['is_consumption'] == is_consumption)]
        
#         print(len(input_df_subset))

        n_models = 7
        
        if scaled:
            scale_factor = input_df_subset['installed_capacity'] / 1000
        else:
            scale_factor = 1
        

        features = select_features(raw_features, weather_features, client_features, is_consumption=is_consumption, target=keys[0][1], verbose=0)
        boosters = get_boosters(keys[is_consumption*n_models + 0], i)
        pred_0 = np.array([model.predict(input_df_subset[features]) for model in boosters]).mean(axis=0) * scale_factor
        
        features = select_features(raw_features, weather_features, client_features, is_consumption=is_consumption, target=keys[1][1], verbose=0)
        boosters = get_boosters(keys[is_consumption*n_models + 1], i)
        pred_pos = np.array([model_pos.predict(input_df_subset[features]) for model_pos in boosters]).mean(axis=0)
        boosters = get_boosters(keys[is_consumption*n_models + 2], i)
        pred_neg = np.array([model_neg.predict(input_df_subset[features]) for model_neg in boosters]).mean(axis=0)
        
        pred_1 = (pred_pos - pred_neg) * scale_factor + input_df_subset['lag_7_day_mean']
        
        features = select_features(raw_features, weather_features, client_features, is_consumption=is_consumption, target=keys[3][1], verbose=0)
        boosters = get_boosters(keys[is_consumption*n_models + 3], i)
        pred_pos = np.array([model_pos.predict(input_df_subset[features]) for model_pos in boosters]).mean(axis=0)
        boosters = get_boosters(keys[is_consumption*n_models + 4], i)
        pred_neg = np.array([model_neg.predict(input_df_subset[features]) for model_neg in boosters]).mean(axis=0)
        
        pred_2 = (pred_pos - pred_neg) * scale_factor + input_df_subset['lag_2_day']
        
        features = select_features(raw_features, weather_features, client_features, is_consumption=is_consumption, target=keys[5][1], verbose=0)
        boosters = get_boosters(keys[is_consumption*n_models + 5], i)
        pred_pos = np.array([model_pos.predict(input_df_subset[features]) for model_pos in boosters]).mean(axis=0)
        boosters = get_boosters(keys[is_consumption*n_models + 6], i)
        pred_neg = np.array([model_neg.predict(input_df_subset[features]) for model_neg in boosters]).mean(axis=0)
        
        pred_3 = (pred_pos - pred_neg) * scale_factor + input_df_subset['lag_7_day']
        
        pred = np.array(pred_0*coef_0 + pred_1*coef_1 + pred_2*coef_2 + pred_3*coef_3).clip(min=0)
        

        
        
        
        input_df.loc[
            (input_df['is_consumption'] == is_consumption),
#                 * (test['is_business'] == is_business), 
            'pred'] = pred
        
    return [pred_0, pred_1, pred_2, pred_3, input_df]

In [47]:
for is_consumption in [0, 1]:

    train_subset = train[
        (train['is_consumption'] == is_consumption)
    #             * (train['is_business'] == is_business)
    ]


    train_subset.reset_index(drop=True, inplace=True)



    X = train_subset[features]
    Y = train_subset['target']

    G = train_subset['month']


    error_dfs = []
    MAEs = []

    MAE_by_shift = [[], [], [], [], []]

    for i, (train_index, test_index) in enumerate(gkf.split(X, Y, G)):
        index = test_index

        preds = get_prediciton(train_subset.iloc[index], results, is_consumptions=[is_consumption], i=i, scaled=False)

        preds[-1] = preds[-1]['pred']

        targets = train_subset.iloc[index]['target']


    #     pred = preds.mean(axis=0)

        print(f"MAE_SCORE:", end='')
        for j, pred in enumerate(preds):

            mae = abs(targets - pred)
            MAE_by_shift[j].append(mae.mean())
            print(f"\t{mae.mean():.2f}", end='')

        print('')

        MAEs.append(mae.mean())

        error_df = train_subset.iloc[index][features+['month']].copy()
        error_df['MAE'] = mae
        error_df['ME'] = pred - targets

        error_dfs.append(error_df)

    print(f'mean mea: {np.mean(MAEs):.2f}')
    agg_error_df = pd.concat(error_dfs)



Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
MAE_SCORE:	44.53	44.37	49.88	48.54	42.09
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
MAE_SCORE:	22.50	23.01	23.12	24.16	21.43
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
MAE_SCORE:	16.16	16.49	16.20	18.27	15.61
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
MAE_SCORE:	6.21	7.14	7.49	7.78	6.56
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
MAE_SCORE:	20.45	20.86	22.09	22.05	19.70
mean mea: 21.08
MAE_SCORE:	49.16	48.52	49.49	48.06	44.44
MAE_SCORE:	55.42	56.50	55.30	57.28	50.97
MAE_SCORE:	60.88	59.40	54.17	65.50	55.05
MAE_SCORE:	36.74	36.75	36.88	33.91	33.18


In [48]:
np.mean(MAE_by_shift, axis=1)

array([48.65867743, 48.48508748, 47.55686376, 49.31105892, 44.24566846])

In [49]:
   
for ft in ['county', 'product_type', 'daytype', 'dayofweek', 'month']:
# for ft in ['county', 'is_consumption', 'is_business', 'product_type',]:
    plt.figure(figsize=(9,4))
    sns.boxenplot(
        agg_error_df[[ft, 'ME']], x=ft, y='ME', outlier_prop=0.03,showfliers=False, k_depth='proportion',
    )
    plt.show()

    print(f"feature: {ft}")
    for feature_value in agg_error_df[ft].unique():
        me = agg_error_df[agg_error_df[ft]==feature_value]['ME'].mean()
        mae = abs(agg_error_df[agg_error_df[ft]==feature_value]['ME']).mean()
        print(f"\tfeature_value: {feature_value}, ME {me:.2f}, MAE {mae:.2f}")
            


<Figure size 900x400 with 1 Axes>

feature: county
	feature_value: 0, ME -12.32, MAE 103.63
	feature_value: 1, ME 0.13, MAE 5.83
	feature_value: 2, ME -10.12, MAE 66.94
	feature_value: 3, ME -1.44, MAE 22.79
	feature_value: 4, ME -2.57, MAE 37.55
	feature_value: 5, ME -3.59, MAE 44.87
	feature_value: 6, ME -1.11, MAE 24.15
	feature_value: 7, ME -2.45, MAE 45.68
	feature_value: 8, ME 0.25, MAE 12.29
	feature_value: 9, ME -0.12, MAE 13.72
	feature_value: 10, ME -1.69, MAE 27.02
	feature_value: 11, ME -7.37, MAE 78.30
	feature_value: 12, ME -4.79, MAE 48.09
	feature_value: 13, ME -3.68, MAE 20.78
	feature_value: 14, ME -3.65, MAE 48.77
	feature_value: 15, ME -1.04, MAE 24.79


<Figure size 900x400 with 1 Axes>

feature: product_type
	feature_value: 1, ME -0.45, MAE 15.66
	feature_value: 2, ME -1.09, MAE 8.01
	feature_value: 3, ME -7.69, MAE 71.24
	feature_value: 0, ME -3.77, MAE 64.33


<Figure size 900x400 with 1 Axes>

feature: daytype
	feature_value: holiday, ME 0.65, MAE 39.79
	feature_value: post_holiday, ME -5.08, MAE 47.19
	feature_value: inner_weekday, ME -6.73, MAE 46.50
	feature_value: pre_holiday, ME -5.25, MAE 45.94


<Figure size 900x400 with 1 Axes>

feature: dayofweek
	feature_value: 6, ME 0.46, MAE 37.46
	feature_value: 0, ME -5.33, MAE 46.77
	feature_value: 1, ME -7.15, MAE 48.49
	feature_value: 2, ME -5.38, MAE 47.25
	feature_value: 3, ME -6.67, MAE 46.32
	feature_value: 4, ME -3.60, MAE 45.33
	feature_value: 5, ME -0.81, MAE 39.93


<Figure size 900x400 with 1 Axes>

feature: month
	feature_value: 5, ME -8.73, MAE 46.77
	feature_value: 7, ME -4.99, MAE 37.87
	feature_value: 8, ME -16.86, MAE 46.27
	feature_value: 2, ME 3.10, MAE 45.96
	feature_value: 3, ME 0.01, MAE 54.53
	feature_value: 6, ME 9.37, MAE 52.96
	feature_value: 9, ME -18.99, MAE 56.69
	feature_value: 1, ME -14.36, MAE 53.50
	feature_value: 10, ME -1.69, MAE 26.09
	feature_value: 11, ME -2.20, MAE 40.44
	feature_value: 12, ME 3.05, MAE 26.18
	feature_value: 4, ME 3.55, MAE 49.14


In [50]:
import lightgbm

for key in results.keys():
    print(key)
    importance_split = np.mean([booster.feature_importance(importance_type='split') / booster.feature_importance(importance_type='split').sum()
                     for booster in results[key]['cvbooster'].boosters], axis=0) * 100
    
    importance_gain = np.mean([booster.feature_importance(importance_type='gain') / booster.feature_importance(importance_type='gain').sum()
                     for booster in results[key]['cvbooster'].boosters], axis=0) * 100
    
    importance = importance_gain / importance_split
    
    feature_name = results[key]['cvbooster'].boosters[0].feature_name()
    
#     for imp in [importance_split, importance_gain, importance]:
    for imp in [importance_gain]:
        tuples = sorted(zip(feature_name, imp), key=lambda x: x[1], reverse=True)

        plt.figure(figsize=(10,8))
        ax = sns.barplot(pd.DataFrame(tuples), x=1, y=0, orient='h')
        for i in ax.containers:
            ax.bar_label(i,)
        plt.show()

(0, 'target')


<Figure size 1000x800 with 1 Axes>

(0, 'lag_7_day_mean_diff')


<Figure size 1000x800 with 1 Axes>

(0, 'lag_7_day_mean_diff_neg')


...

<Figure size 1000x800 with 1 Axes>

(1, 'lag_7_day_diff')


<Figure size 1000x800 with 1 Axes>

(1, 'lag_7_day_diff_neg')


<Figure size 1000x800 with 1 Axes>

In [51]:
agg_error_df['datetime'] = train[train['is_consumption']==1]['datetime'].values
agg_error_df['hour'] = train[train['is_consumption']==1]['hour'].values
agg_error_df['date'] = train[train['is_consumption']==1]['date'].values

In [52]:
sns.boxenplot(
    agg_error_df[['hour', 'ME']], x='hour', y='ME', outlier_prop=0.03,showfliers=False, k_depth='proportion',
)
plt.show()

<Figure size 640x480 with 1 Axes>

In [53]:
plt.figure(figsize=(14,8))
sns.scatterplot(
    agg_error_df[['date', 'ME']].groupby('date').mean().reset_index(), x='date', y='ME', marker='.')

<Axes: xlabel='date', ylabel='ME'>

<Figure size 1400x800 with 1 Axes>

In [54]:
plt.figure(figsize=(14,8))
sns.scatterplot(
    agg_error_df[['datetime', 'MAE']], x='datetime', y='MAE', marker='.')

<Axes: xlabel='datetime', ylabel='MAE'>

<Figure size 1400x800 with 1 Axes>

# Make pred & Submission

In [55]:
inspect = []


def append_lag(test, placeholder_previous_day, historic_train, predicting_start_time, truncate=True, client_df=None):
#     inspect.append(test)
    
    new_df = pd.concat([test, placeholder_previous_day, historic_train,])
    new_df, _ = fill_missing_time(new_df, start_time=placeholder_previous_day['datetime'].min()-datetime.timedelta(days=30), inference=True)
    
    
    new_df = append_daytype(new_df)

    del new_df['eic_count']
    del new_df['installed_capacity']
    del new_df['available_date']
    
    if client_df is not None:
#         pass
        new_df = new_df.merge(
            full_client.drop("date", axis=1),  
            left_on=["county", "is_business", "product_type", "date"], 
            right_on=["county", "is_business", "product_type", "available_date"], 
            how="left")
    
    new_df = new_df.set_index(['county', 'is_business', 'product_type', 'is_consumption', 'datetime']).sort_index(ascending=True)


    grouped_eic_count = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['eic_count']
#     new_df['eic_count_2_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*0).fillna(0)
#     new_df['eic_count_7_day_diff'] = grouped_eic_count.shift(0).fillna(0) - grouped_eic_count.shift(24*5).fillna(0)
#     new_df['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0).fillna(0) - np.mean([grouped_eic_count.shift(24*i).fillna(0).values for i in range(0, 7)], axis=0)

    new_df['eic_count_2_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*0)
    new_df['eic_count_7_day_diff'] = grouped_eic_count.shift(0) - grouped_eic_count.shift(24*5)
    new_df['eic_count_7_day_mean_diff'] = grouped_eic_count.shift(0) - np.mean([grouped_eic_count.shift(24*i).values for i in range(0, 7)], axis=0)

    
    grouped_installed_capacity = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['installed_capacity']
#     new_df['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*0).fillna(0)
#     new_df['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0).fillna(0) - grouped_installed_capacity.shift(24*5).fillna(0)
#     new_df['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0).fillna(0) - np.mean([grouped_installed_capacity.shift(24*i).fillna(0) for i in range(0, 7)], axis=0)

    new_df['installed_capacity_2_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*0)
    new_df['installed_capacity_7_day_diff'] = grouped_installed_capacity.shift(0) - grouped_installed_capacity.shift(24*5)
    new_df['installed_capacity_7_day_mean_diff'] = grouped_installed_capacity.shift(0) - np.mean([grouped_installed_capacity.shift(24*i) for i in range(0, 7)], axis=0)

    
    
    grouped_target = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target']
    
    for i in range(2, 17):
        new_df[f'lag_{i}_day'] = grouped_target.shift(24*i).fillna(0)

    
    new_df['lag_7_day_mean'] = np.mean([new_df[f'lag_{i}_day'] for i in range(2,9)], axis=0) 
    new_df['lag_8_14_day_mean'] = np.mean([new_df[f'lag_{i}_day'] for i in range(9,16)], axis=0)

    new_df['lag_2_day_1h'] = grouped_target.shift(48+1)
    new_df['lag_3_day_1h'] = grouped_target.shift(72+1)
    new_df['lag_4_day_1h'] = grouped_target.shift(96+1)
    
    new_df['rolling_mean_24'] = new_df.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).mean().shift(base_lag).values
    new_df['rolling_std_24'] = new_df.reset_index().groupby(['county', 'is_business', 'product_type', 'is_consumption'])['target'].rolling(24).std().shift(base_lag).values

    
    grouped_consumption_counterpart = new_df.groupby(['county', 'is_business', 'product_type', 'is_consumption'])['consumption_counterpart']
    for i in range(2, 9):
        new_df[f'lag_{i}_day_consumption_counterpart'] = grouped_target.shift(24*i)

    new_df['lag_7_day_mean_consumption_counterpart'] = np.mean([new_df[f'lag_{i}_day_consumption_counterpart'] for i in range(2,9)], axis=0)
    
    new_df = new_df.reset_index()
    if truncate:
        new_df = new_df[new_df['datetime'] >= predicting_start_time]
    
        return new_df[new_df['row_id'].notna()]

    return new_df

In [56]:
import enefit
env = enefit.make_env()

# Check the datetime of the test dataframe

In [57]:
def catogorical_to_int_type(df):
    df['is_business'] = df['is_business'].astype(int)
    df['is_consumption'] = df['is_consumption'].astype(int)
    df['county'] = df['county'].astype(int)
    df['product_type'] = df['product_type'].astype(int)
    
    return df

In [58]:
history_cutoff = "2023-04-28 00:00:00"


debug = False
if debug:
    enefit.make_env.__called__ = False
    type(env)._state = type(type(env)._state).__dict__['INIT']
    
    
iter_test = env.iter_test()

sample_prediction_last = None
sample_prediction_last_last = None

init = False

for (test, revealed_targets, client, historical_weather,
        forecast_weather, electricity_prices, gas_prices, sample_prediction) in iter_test:
    
#     assert False
    if sample_prediction_last_last is not None:
        print("MAE: ", abs(revealed_targets['target'] - sample_prediction_last_last['target']).mean())
    
    
    test=test.rename(columns={'prediction_datetime':'datetime'})
    
    test['date'] = pd.to_datetime(test.datetime).dt.date
    
    predicting_start_time = test['datetime'].min()
    print(f"Predicting start time: {predicting_start_time}")
    
    test = catogorical_to_int_type(test)



    # Electricity Prices Data
    # TODO: check data
    if not init:
        cutoff_time = electricity_prices['forecast_date'].min()
        print(electricity_prices['forecast_date'].min())
        full_elec_prices = elec_prices[
            (pd.to_datetime(elec_prices['forecast_date']) >= pd.to_datetime(history_cutoff)) 
            * (pd.to_datetime(elec_prices['forecast_date']) < cutoff_time)
        ]
    full_elec_prices = pd.concat([full_elec_prices, electricity_prices])
    test = test.merge(
        process_elec_prices(full_elec_prices)[["euros_per_mwh", 
                                               'euros_per_mwh_lag_7_day_mean_diff', 
                                               "daily_euros_per_mwh", 
                                                "euros_per_mwh_lag_7_day_diff",
                                                "euros_per_mwh_lag_2_day_diff",
                                               "data_available_datetime",]],
        how="left",
        right_on="data_available_datetime",
        left_on="datetime",
       )
    
    # Client Data
    client['date'] = pd.to_datetime(client.date).dt.date
    client['available_date'] = client['date'] + datetime.timedelta(days=2)
    client['is_business'] = client['is_business'].astype(int)
    client['product_type'] = client['product_type'].astype(int)
    client['county'] = client['county'].astype(int)
    
    if not init:
        cutoff_time = client['available_date'].min()
        full_client = client_train[
            (pd.to_datetime(client_train['available_date']) >= pd.to_datetime(history_cutoff))
            * (pd.to_datetime(client_train['available_date']) < pd.to_datetime(cutoff_time))
        ]
    full_client = pd.concat([full_client, client])

#     test = test.merge(
#         full_client.drop("date", axis=1),  
#         left_on=["county", "is_business", "product_type", "date"], 
#         right_on=["county", "is_business", "product_type", "available_date"], 
#         how="left")
    
    historical_weather['datetime'] = pd.to_datetime(historical_weather['datetime'])
    if not init:
        cutoff_time = historical_weather['datetime'].min()
        print(historical_weather['datetime'].min())
        full_historical_weather = historical_weather_train[
            (pd.to_datetime(historical_weather_train['datetime']) >= pd.to_datetime(history_cutoff)) 
            * (pd.to_datetime(historical_weather_train['datetime']) < cutoff_time)
        ]
    full_historical_weather = pd.concat([full_historical_weather, historical_weather])
    
    forecast_weather['forecast_datetime'] = pd.to_datetime(forecast_weather['forecast_datetime'])
    if not init:
        cutoff_time = forecast_weather['forecast_datetime'].min()
        print(forecast_weather['forecast_datetime'].min())
        full_forecast_weather = forecast_weather_train[
            (forecast_weather_train['forecast_datetime'] >= pd.to_datetime(history_cutoff)) 
            * (forecast_weather_train['forecast_datetime']< cutoff_time)
        ]
    full_forecast_weather = pd.concat([full_forecast_weather, forecast_weather])
    
    test = test.merge(
        get_weighted_forecast_by_county(full_forecast_weather,  get_weighted_historical_weather_by_county(full_historical_weather)),
        how="left",
        on=["county", "datetime",],
    )

    

    test["month"]  = test["datetime"].dt.month

    test['hour'] = test['datetime'].dt.hour
    test['sin_hour'] = np.sin(2*np.pi*test.hour/hours_in_day)
    test['cos_hour'] = np.cos(2*np.pi*test.hour/hours_in_day)
    test['hour'] = test['hour'].astype(dtype)
    
    test['sin_month'] = np.sin(2*np.pi*test.month/months_in_year)
    test['cos_month'] = np.cos(2*np.pi*test.month/months_in_year)

    if not init:
        cutoff_time = revealed_targets['datetime'].min()
        print(revealed_targets['datetime'].min())
        historic_train = train[
            (train['datetime'] >= pd.to_datetime(history_cutoff)) * (train['datetime'] < cutoff_time)
        ]
    historic_train = pd.concat([historic_train, revealed_targets])
    
    placeholder_previous_day = revealed_targets.copy()
    placeholder_previous_day['datetime'] += datetime.timedelta(days=1)
    placeholder_previous_day['target'] = np.nan
    
    
    test['target'] = np.nan
    
#     inspect = append_lag(test, placeholder_previous_day, historic_train, predicting_start_time=predicting_start_time, client_df=full_client, truncate=False)
#     assert False
    
    test = append_lag(test, placeholder_previous_day, historic_train, predicting_start_time=predicting_start_time, client_df=full_client)

    test['county'] = test['county'].astype(dtype)
    test['product_type'] = test['product_type'].astype(dtype)
    test['dayofweek'] = test['datetime'].dt.dayofweek.astype(dtype)
    

    test = test.sort_values('row_id', ascending=True)
    
    preds = get_prediciton(test, results, scaled=False)
    test = preds[-1]

    sample_prediction['target'] = test['pred'].reset_index(drop=True).fillna(0).values.astype(float)
    
    sample_prediction_last_last = sample_prediction_last
    sample_prediction_last = sample_prediction
    
    print(f"predicting {len(sample_prediction)} rows")
    env.predict(sample_prediction)
    
    init = True

    

This version of the API is not optimized and should not be used to estimate the runtime of your code on the hidden test set.
Predicting start time: 2023-05-28 00:00:00
2023-05-27 00:00:00
2023-05-26 11:00:00
2023-05-27 03:00:00
2023-05-26 00:00:00
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
predicting 3120 rows
Predicting start time: 2023-05-29 00:00:00
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
predicting 3120 rows
MAE:  32.947989896436376
Predicting start time: 2023-05-30 00:00:00
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
predicting 3120 rows
MAE:  47.92201573033418
Predicting start time: 2023-05-31 00:00:00
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
Removing aborted: euros_per_mwh_lag_7_day_mean_diff not found
predicting 3120

In [59]:
# inspect[
#     inspect['datetime'].dt.hour == 0 
# ][['datetime', 'daytype', 'daytype_lag_2_day', 'daytype_lag_7_day']].tail(50)

In [60]:
# inspect.iloc[:, 90:].tail(30)

In [61]:
test.isna().sum(axis=0)[test.isna().sum(axis=0) > 0]

target                                  3120
data_block_id_x                         3120
euros_per_mwh_lag_28_day_mean_diff      3120
consumption_counterpart                 3120
lag_7_day_1h                            3120
rolling_mean_24x7x4                     3120
lag_2_day_1h_consumption_counterpart    3120
day                                     3120
yearmonth                               3120
school_holiday                          3120
weight                                  3120
lag_7_day_mean_diff                     3120
lag_7_day_mean_diff_neg                 3120
lag_2_day_diff                          3120
lag_2_day_diff_neg                      3120
lag_7_day_diff                          3120
lag_7_day_diff_neg                      3120
data_block_id_y                         3120
dtype: int64

In [62]:
test.isna().sum(axis=0)[test.isna().sum(axis=0) == 0]

county            0
is_business       0
product_type      0
is_consumption    0
datetime          0
                 ..
lag_12_day        0
lag_13_day        0
lag_14_day        0
lag_15_day        0
pred              0
Length: 134, dtype: int64